# 02_mvp_aapl_date_split

Исправленная MVP-версия notebook после первого запуска `01_mvp_aapl_check`.

Что исправлено относительно версии 01:

- split делается по **уникальным `decision_date`**, поэтому одна дата не может попасть одновременно в validation и test;
- parser распознаёт header таблицы вида `date,open,high,...`, поэтому числовые признаки получают реальные имена;
- сохраняется `model_index.csv` со split label для каждой строки;
- добавлены assert-проверки отсутствия пересечения дат между train / validation / test.

Notebook нужно запустить сверху вниз заново. Результаты из версии 01 не считаются финальными из-за пересечения дат в split.


In [ ]:
# Cell 1. Imports, config, reproducibility

import os
import re
import json
import random
from pathlib import Path
from datetime import datetime, timedelta

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel

import yfinance as yf

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_recall_fscore_support,
    roc_auc_score,
    confusion_matrix,
    classification_report,
)
from torch.optim.lr_scheduler import ReduceLROnPlateau

SEED = 42
TICKER = "aapl"
TICKER_YF = TICKER.upper()
MAX_DAYS = 10
EMB_DIM = 768

# Conservative mode: use texts dated up to t-1 if intraday timestamps are unavailable.
TEXT_LAG_DAYS = 1

TRAIN_SIZE = 0.70
VAL_SIZE = 0.15
TEST_SIZE = 0.15

BATCH_SIZE = 8
EPOCHS = 50
PATIENCE = 7
LR = 1e-3

OUTPUT_DIR = Path("reports/tables")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
print("Ticker:", TICKER_YF)

In [ ]:
# Cell 2. Load FinBERT

model_name = "ProsusAI/finbert"

tokenizer = AutoTokenizer.from_pretrained(model_name)
finbert = AutoModel.from_pretrained(model_name).to(device)
finbert.eval()

print("FinBERT loaded:", model_name)

In [ ]:
# Cell 3. Load text dataset and extract decision_date

dataset = load_dataset("TheFinAI/flare-sm-bigdata", split="train")
df = dataset.to_pandas()

def extract_decision_date(query: str):
    if not isinstance(query, str):
        return None
    match = re.search(r"at (\d{4}-\d{2}-\d{2})", query)
    if match:
        return datetime.strptime(match.group(1), "%Y-%m-%d").date()
    return None

df_ticker = df[df["query"].str.contains(rf"\b{TICKER}\b", case=False, na=False)].copy()
df_ticker["decision_date"] = df_ticker["query"].apply(extract_decision_date)
df_ticker = (
    df_ticker
    .dropna(subset=["decision_date"])
    .sort_values("decision_date")
    .reset_index(drop=True)
)

print("Rows for ticker:", len(df_ticker))
print("Date range:", df_ticker["decision_date"].min(), "->", df_ticker["decision_date"].max())
display(df_ticker[["query", "decision_date"]].head())

In [ ]:
# Cell 4. Load market data and create next-day target

min_date = df_ticker["decision_date"].min() - timedelta(days=60)
max_date = df_ticker["decision_date"].max() + timedelta(days=10)

stock_data = yf.download(TICKER_YF, start=min_date, end=max_date, auto_adjust=False, progress=False)

if stock_data.empty:
    raise RuntimeError(f"No market data loaded for {TICKER_YF}.")

# yfinance can return MultiIndex columns depending on version / request.
if isinstance(stock_data.columns, pd.MultiIndex):
    if TICKER_YF in stock_data.columns.get_level_values(-1):
        stock_data = stock_data.xs(TICKER_YF, axis=1, level=-1)
    else:
        stock_data.columns = stock_data.columns.get_level_values(0)

required_columns = ["Open", "High", "Low", "Close", "Adj Close", "Volume"]
available_columns = [col for col in required_columns if col in stock_data.columns]
stock_data = stock_data[available_columns].copy()

stock_data = stock_data.reset_index()
stock_data["Date"] = pd.to_datetime(stock_data["Date"]).dt.date

# Formal MVP target: y_t = 1{Close_{t+1} > Close_t}
stock_data["next_close"] = stock_data["Close"].shift(-1)
stock_data["target"] = (stock_data["next_close"] > stock_data["Close"]).astype("float")
stock_data.loc[stock_data["next_close"].isna(), "target"] = np.nan
stock_data = stock_data.dropna(subset=["target"]).copy()
stock_data["target"] = stock_data["target"].astype(int)

print("Market rows:", len(stock_data))
print("Market date range:", stock_data["Date"].min(), "->", stock_data["Date"].max())
display(stock_data.head())

In [ ]:
# Cell 5. Text/table parsing helpers

def parse_text_and_table(raw_text: str):
    """
    Split raw FLARE text into:
    - table header names;
    - table rows;
    - dictionary of texts by date.

    Supported table header formats:
    1. Context: date,open,high,low,...
    2. date,open,high,low,...
    """
    if not isinstance(raw_text, str):
        return [], [], {}

    lines = raw_text.strip().split("\n")
    table_header = []
    table_lines = []
    text_dict = {}
    in_table = True

    for line in lines:
        stripped = line.strip()
        if not stripped:
            continue

        lowered = stripped.lower()

        if lowered.startswith("context:"):
            header_raw = stripped.split(":", 1)[1].strip()
            table_header = [x.strip() for x in header_raw.split(",")]
            continue

        # FLARE rows can start directly with the header:
        # date,open,high,low,close,adj-close,inc-5,...
        if not table_header and lowered.startswith("date,"):
            table_header = [x.strip() for x in stripped.split(",")]
            continue

        if in_table:
            if re.match(r"\d{4}-\d{2}-\d{2},", stripped):
                table_lines.append(stripped)
            elif re.match(r"\d{4}-\d{2}-\d{2}:\s", stripped):
                in_table = False
                date_part, text_part = stripped.split(":", 1)
                date_str = date_part.strip()
                text_dict.setdefault(date_str, []).append(text_part.strip())
        else:
            if re.match(r"\d{4}-\d{2}-\d{2}:\s", stripped):
                date_part, text_part = stripped.split(":", 1)
                date_str = date_part.strip()
                text_dict.setdefault(date_str, []).append(text_part.strip())

    return table_header, table_lines, text_dict


def safe_parse_date(date_str: str):
    try:
        return datetime.strptime(date_str, "%Y-%m-%d").date()
    except Exception:
        return None


def get_finbert_embedding(text: str) -> torch.Tensor:
    if not isinstance(text, str) or not text.strip():
        return torch.zeros(EMB_DIM)

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=128,
        padding=False,
    ).to(device)

    with torch.no_grad():
        outputs = finbert(**inputs)

    return outputs.last_hidden_state[:, 0, :].detach().cpu().squeeze(0)


def parse_text_to_features(
    raw_text: str,
    decision_date,
    max_days: int = MAX_DAYS,
    text_lag_days: int = TEXT_LAG_DAYS,
):
    """
    Returns:
    - tensor with shape (max_days, EMB_DIM + n_numeric_features);
    - info dict for audit.

    Numeric table rows are allowed up to decision_date.
    Texts are allowed up to decision_date - text_lag_days.
    """
    table_header, table_lines, text_dict = parse_text_and_table(raw_text)

    numeric_feature_names = table_header[1:] if table_header and table_header[0].lower() == "date" else []

    text_cutoff = decision_date - timedelta(days=text_lag_days)

    data_entries = []
    for line in table_lines:
        parts = [x.strip() for x in line.split(",")]
        if len(parts) < 2:
            continue

        row_date = safe_parse_date(parts[0])
        if row_date is None or row_date > decision_date:
            continue

        try:
            nums = [float(x) for x in parts[1:]]
        except ValueError:
            continue

        # Fallback only if there was genuinely no header.
        if not numeric_feature_names:
            numeric_feature_names = [f"numeric_{i}" for i in range(len(nums))]

        if len(nums) != len(numeric_feature_names):
            continue

        data_entries.append({"date": row_date, "nums": nums})

    data_entries.sort(key=lambda x: x["date"])
    data_entries = data_entries[-max_days:]

    n_num = len(numeric_feature_names)
    num_features_list = []
    emb_list = []
    used_dates = []
    used_text_count = 0

    for entry in data_entries:
        row_date = entry["date"]
        date_str = row_date.isoformat()

        texts_for_date = []
        if row_date <= text_cutoff:
            texts_for_date = text_dict.get(date_str, [])

        text_joined = " ".join(texts_for_date)
        emb = get_finbert_embedding(text_joined) if text_joined else torch.zeros(EMB_DIM)

        used_text_count += len(texts_for_date)
        used_dates.append(date_str)
        num_features_list.append(entry["nums"])
        emb_list.append(emb)

    pad_len = max_days - len(data_entries)
    for _ in range(pad_len):
        num_features_list.insert(0, [0.0] * n_num)
        emb_list.insert(0, torch.zeros(EMB_DIM))
        used_dates.insert(0, None)

    combined = []
    for nums, emb in zip(num_features_list, emb_list):
        combined.append(torch.cat([emb, torch.tensor(nums, dtype=torch.float32)]))

    info = {
        "numeric_feature_names": numeric_feature_names,
        "n_numeric_features": n_num,
        "valid_table_rows": len(data_entries),
        "used_dates": used_dates,
        "used_text_count": used_text_count,
        "text_cutoff": text_cutoff.isoformat(),
    }

    if n_num == 0:
        return None, info

    return torch.stack(combined), info


# Quick parser smoke test on the first row.
header_sample, table_sample, text_sample = parse_text_and_table(df_ticker["text"].iloc[0])
print("Parsed table header:", header_sample)
print("First table row:", table_sample[0] if table_sample else None)
print("First text date:", next(iter(text_sample.keys())) if text_sample else None)


In [ ]:
# Cell 6. Build modeling dataset

X_list = []
y_list = []
date_list = []
info_list = []

skipped_no_price = 0
skipped_no_features = 0
skipped_inconsistent_numeric_dim = 0

numeric_feature_names_ref = None

for _, row in df_ticker.iterrows():
    decision_date = row["decision_date"]

    price_row = stock_data.loc[stock_data["Date"] == decision_date]
    if price_row.empty:
        skipped_no_price += 1
        continue

    target = int(price_row["target"].iloc[0])

    feat_tensor, info = parse_text_to_features(
        row["text"],
        decision_date=decision_date,
        max_days=MAX_DAYS,
        text_lag_days=TEXT_LAG_DAYS,
    )

    if feat_tensor is None or info["valid_table_rows"] == 0:
        skipped_no_features += 1
        continue

    if numeric_feature_names_ref is None:
        numeric_feature_names_ref = info["numeric_feature_names"]

    if info["numeric_feature_names"] != numeric_feature_names_ref:
        skipped_inconsistent_numeric_dim += 1
        continue

    X_list.append(feat_tensor)
    y_list.append(target)
    date_list.append(decision_date)
    info_list.append(info)

if not X_list:
    raise RuntimeError("No training examples left. Check ticker, date alignment and parser.")

X = torch.stack(X_list)
y = torch.tensor(y_list, dtype=torch.float32)
dates = np.array(date_list)

df_model_index = pd.DataFrame({
    "decision_date": dates,
    "target": y.numpy().astype(int),
    "valid_table_rows": [info["valid_table_rows"] for info in info_list],
    "used_text_count": [info["used_text_count"] for info in info_list],
    "text_cutoff": [info["text_cutoff"] for info in info_list],
}).sort_values("decision_date").reset_index(drop=True)

# Ensure X/y are sorted by decision_date.
order = np.argsort(dates)
X = X[order]
y = y[order]
dates = dates[order]

print("Total text rows:", len(df_ticker))
print("Skipped no price:", skipped_no_price)
print("Skipped no features:", skipped_no_features)
print("Skipped inconsistent numeric dim:", skipped_inconsistent_numeric_dim)
print("Final X shape:", tuple(X.shape))
print("Class counts:", pd.Series(y.numpy().astype(int)).value_counts().to_dict())
print("Numeric feature names:", numeric_feature_names_ref)

display(df_model_index.head())

In [ ]:
# Cell 7. Chronological train/validation/test split by unique decision_date and scaler fit only on train

X_np = X.numpy()
y_np = y.numpy().astype(int)

unique_dates = np.array(sorted(pd.Series(dates).unique()))
n_unique_dates = len(unique_dates)

train_date_end = int(n_unique_dates * TRAIN_SIZE)
val_date_end = int(n_unique_dates * (TRAIN_SIZE + VAL_SIZE))

if train_date_end <= 0 or val_date_end <= train_date_end or val_date_end >= n_unique_dates:
    raise RuntimeError(
        f"Invalid date split sizes for n_unique_dates={n_unique_dates}: "
        f"train_date_end={train_date_end}, val_date_end={val_date_end}"
    )

train_dates = set(unique_dates[:train_date_end])
val_dates = set(unique_dates[train_date_end:val_date_end])
test_dates = set(unique_dates[val_date_end:])

assert train_dates.isdisjoint(val_dates)
assert train_dates.isdisjoint(test_dates)
assert val_dates.isdisjoint(test_dates)

train_mask = np.array([d in train_dates for d in dates])
val_mask = np.array([d in val_dates for d in dates])
test_mask = np.array([d in test_dates for d in dates])

assert not np.any(train_mask & val_mask)
assert not np.any(train_mask & test_mask)
assert not np.any(val_mask & test_mask)
assert (train_mask | val_mask | test_mask).all()

X_train, y_train, dates_train = X_np[train_mask], y_np[train_mask], dates[train_mask]
X_val, y_val, dates_val = X_np[val_mask], y_np[val_mask], dates[val_mask]
X_test, y_test, dates_test = X_np[test_mask], y_np[test_mask], dates[test_mask]

n_numeric = X_np.shape[2] - EMB_DIM
if n_numeric <= 0:
    raise RuntimeError(f"Expected numeric features after first {EMB_DIM} embedding dims.")

scaler = StandardScaler()

def fit_transform_numeric_train(X_arr):
    X_out = X_arr.copy()
    numeric = X_out[:, :, EMB_DIM:].reshape(-1, n_numeric)
    scaler.fit(numeric)
    X_out[:, :, EMB_DIM:] = scaler.transform(numeric).reshape(X_out.shape[0], X_out.shape[1], n_numeric)
    return X_out

def transform_numeric(X_arr):
    X_out = X_arr.copy()
    numeric = X_out[:, :, EMB_DIM:].reshape(-1, n_numeric)
    X_out[:, :, EMB_DIM:] = scaler.transform(numeric).reshape(X_out.shape[0], X_out.shape[1], n_numeric)
    return X_out

X_train = fit_transform_numeric_train(X_train)
X_val = transform_numeric(X_val)
X_test = transform_numeric(X_test)

# Save row-level split audit.
split_labels = np.empty(len(dates), dtype=object)
split_labels[train_mask] = "train"
split_labels[val_mask] = "validation"
split_labels[test_mask] = "test"

model_index_df = pd.DataFrame({
    "decision_date": dates,
    "target": y_np,
    "split": split_labels,
})
model_index_df.to_csv(OUTPUT_DIR / "model_index.csv", index=False)

dataset_summary = {
    "ticker": TICKER_YF,
    "target": "y_t = 1{Close_{t+1} > Close_t}",
    "frequency": "daily",
    "horizon": "1 trading day",
    "split_type": "chronological_by_unique_decision_date",
    "text_lag_days": TEXT_LAG_DAYS,
    "max_days_window": MAX_DAYS,
    "embedding_model": model_name,
    "embedding_dim": EMB_DIM,
    "numeric_feature_count": n_numeric,
    "numeric_feature_names": numeric_feature_names_ref,
    "n_observations": int(len(X_np)),
    "n_unique_dates": int(n_unique_dates),
    "train_size": int(len(X_train)),
    "validation_size": int(len(X_val)),
    "test_size": int(len(X_test)),
    "train_unique_dates": int(len(train_dates)),
    "validation_unique_dates": int(len(val_dates)),
    "test_unique_dates": int(len(test_dates)),
    "date_min": str(dates.min()),
    "date_max": str(dates.max()),
    "train_date_min": str(dates_train.min()),
    "train_date_max": str(dates_train.max()),
    "validation_date_min": str(dates_val.min()),
    "validation_date_max": str(dates_val.max()),
    "test_date_min": str(dates_test.min()),
    "test_date_max": str(dates_test.max()),
    "skipped_no_price": int(skipped_no_price),
    "skipped_no_features": int(skipped_no_features),
    "skipped_inconsistent_numeric_dim": int(skipped_inconsistent_numeric_dim),
}

pd.DataFrame([dataset_summary]).to_csv(OUTPUT_DIR / "dataset_summary.csv", index=False)

print(json.dumps(dataset_summary, ensure_ascii=False, indent=2))
print("\nSplit row counts:")
print(model_index_df["split"].value_counts().to_string())
print("\nDate overlap checks:")
print("train ∩ validation:", len(train_dates & val_dates))
print("train ∩ test:", len(train_dates & test_dates))
print("validation ∩ test:", len(val_dates & test_dates))


In [ ]:
# Cell 8. Dataset and DataLoaders

class StockDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_dataset = StockDataset(X_train, y_train)
val_dataset = StockDataset(X_val, y_val)
test_dataset = StockDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print("Train/val/test:", len(train_dataset), len(val_dataset), len(test_dataset))

In [ ]:
# Cell 9. Evaluation helpers

def safe_roc_auc(y_true, y_score):
    try:
        if len(np.unique(y_true)) < 2:
            return np.nan
        return roc_auc_score(y_true, y_score)
    except Exception:
        return np.nan


def compute_metrics(model_name, feature_group, y_true, y_pred, y_score=None):
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        average="binary",
        zero_division=0,
    )
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()

    return {
        "model": model_name,
        "feature_group": feature_group,
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "roc_auc": safe_roc_auc(y_true, y_score) if y_score is not None else np.nan,
        "tp": int(tp),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
    }


def flatten_price_features(X_arr):
    return X_arr[:, :, EMB_DIM:].reshape(X_arr.shape[0], -1)


def flatten_llm_features(X_arr):
    return X_arr[:, :, :EMB_DIM].reshape(X_arr.shape[0], -1)


def flatten_combined_features(X_arr):
    return X_arr.reshape(X_arr.shape[0], -1)


metrics_rows = []
prediction_rows = []

In [ ]:
# Cell 10. Baseline models

# Majority baseline: always predict the most frequent class in train.
majority_class = int(pd.Series(y_train).mode().iloc[0])
majority_pred = np.full_like(y_test, majority_class)
majority_score = np.full_like(y_test, fill_value=float(majority_class), dtype=float)

metrics_rows.append(
    compute_metrics("Majority", "none", y_test, majority_pred, majority_score)
)

# Logistic regression baselines.
baseline_specs = [
    ("LogisticRegression", "prices_only", flatten_price_features),
    ("LogisticRegression", "finbert_only", flatten_llm_features),
    ("LogisticRegression", "prices_plus_finbert", flatten_combined_features),
]

for model_label, feature_group, transform_fn in baseline_specs:
    Xtr = transform_fn(X_train)
    Xte = transform_fn(X_test)

    clf = LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=SEED,
        solver="liblinear",
    )
    clf.fit(Xtr, y_train)

    y_pred = clf.predict(Xte)
    y_score = clf.predict_proba(Xte)[:, 1]

    metrics_rows.append(
        compute_metrics(model_label, feature_group, y_test, y_pred, y_score)
    )

    for d, yt, yp, score in zip(dates_test, y_test, y_pred, y_score):
        prediction_rows.append({
            "model": f"{model_label}_{feature_group}",
            "decision_date": d,
            "y_true": int(yt),
            "y_pred": int(yp),
            "score": float(score),
        })

metrics_df = pd.DataFrame(metrics_rows)
display(metrics_df)

In [ ]:
# Cell 11. LSTM model with validation loss and early stopping

class LSTMPredictor(nn.Module):
    def __init__(self, input_dim, hidden_dim=128, num_layers=2, dropout=0.3, bidirectional=True):
        super().__init__()
        self.lstm = nn.LSTM(
            input_dim,
            hidden_dim,
            num_layers,
            batch_first=True,
            bidirectional=bidirectional,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        lstm_out_dim = hidden_dim * 2 if bidirectional else hidden_dim
        self.fc = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(lstm_out_dim, 1),
        )

    def forward(self, x):
        out, _ = self.lstm(x)
        logits = self.fc(out[:, -1, :])
        return logits.squeeze(-1)


input_dim = X_train.shape[2]
model = LSTMPredictor(input_dim=input_dim).to(device)

n_pos = max((y_train == 1).sum(), 1)
n_neg = max((y_train == 0).sum(), 1)
pos_weight = torch.tensor([n_neg / n_pos], dtype=torch.float32, device=device)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=3)

best_val_loss = np.inf
best_state = None
epochs_without_improvement = 0
history = []

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_losses = []

    for xb, yb in train_loader:
        xb = xb.to(device)
        yb = yb.to(device)

        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()

        train_losses.append(loss.item())

    model.eval()
    val_losses = []
    with torch.no_grad():
        for xb, yb in val_loader:
            xb = xb.to(device)
            yb = yb.to(device)
            logits = model(xb)
            loss = criterion(logits, yb)
            val_losses.append(loss.item())

    train_loss = float(np.mean(train_losses))
    val_loss = float(np.mean(val_losses))
    scheduler.step(val_loss)

    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_loss,
    })

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    if epoch % 5 == 0 or epoch == 1:
        print(f"Epoch {epoch:03d}: train_loss={train_loss:.4f}, val_loss={val_loss:.4f}")

    if epochs_without_improvement >= PATIENCE:
        print(f"Early stopping at epoch {epoch}. Best val_loss={best_val_loss:.4f}")
        break

pd.DataFrame(history).to_csv(OUTPUT_DIR / "lstm_loss_history.csv", index=False)

if best_state is not None:
    model.load_state_dict(best_state)

In [ ]:
# Cell 12. LSTM test metrics and save artifacts

model.eval()
lstm_scores = []

with torch.no_grad():
    for xb, _ in test_loader:
        xb = xb.to(device)
        logits = model(xb)
        probs = torch.sigmoid(logits).detach().cpu().numpy()
        lstm_scores.extend(probs)

lstm_scores = np.array(lstm_scores)
lstm_pred = (lstm_scores >= 0.5).astype(int)

metrics_rows.append(
    compute_metrics("LSTM", "prices_plus_finbert", y_test, lstm_pred, lstm_scores)
)

for d, yt, yp, score in zip(dates_test, y_test, lstm_pred, lstm_scores):
    prediction_rows.append({
        "model": "LSTM_prices_plus_finbert",
        "decision_date": d,
        "y_true": int(yt),
        "y_pred": int(yp),
        "score": float(score),
    })

metrics_df = pd.DataFrame(metrics_rows)
predictions_df = pd.DataFrame(prediction_rows)

conf_rows = []
for _, row in metrics_df.iterrows():
    conf_rows.append({
        "model": row["model"],
        "feature_group": row["feature_group"],
        "tp": row["tp"],
        "tn": row["tn"],
        "fp": row["fp"],
        "fn": row["fn"],
    })

confusion_df = pd.DataFrame(conf_rows)

metrics_df.to_csv(OUTPUT_DIR / "metrics.csv", index=False)
confusion_df.to_csv(OUTPUT_DIR / "confusion_matrix.csv", index=False)
predictions_df.to_csv(OUTPUT_DIR / "predictions.csv", index=False)

display(metrics_df)
print("\nClassification report for LSTM:")
print(classification_report(y_test, lstm_pred, target_names=["Fall", "Rise"], zero_division=0))
print("\nSaved files:")
print(OUTPUT_DIR / "dataset_summary.csv")
print(OUTPUT_DIR / "metrics.csv")
print(OUTPUT_DIR / "confusion_matrix.csv")
print(OUTPUT_DIR / "predictions.csv")
print(OUTPUT_DIR / "lstm_loss_history.csv")

## После запуска

Проверьте, что в `dataset_summary.csv`:

- `numeric_feature_names` содержит `open`, `high`, `low`, `close`, `adj-close`, `inc-*`, а не `numeric_0...numeric_10`;
- `split_type = chronological_by_unique_decision_date`;
- `validation_date_max < test_date_min` или, как минимум, нет пересечения дат по split;
- `reports/tables/model_index.csv` содержит split label для каждой строки.
